# STRATEGY: LightGBM with Heavy Regularization
Alternative approach using LightGBM known for better regularization with large datasets

**Why LightGBM:**
- Better handles large datasets (661k rows)
- Different leaf-wise growth (vs depth-wise in XGBoost) - often generalizes better
- Early stopping built-in reduces overfitting
- Better default regularization parameters

**Strategy:**
1. Original 445 features only (no engineering)
2. Heavy regularization: num_leaves=10-15, reg_alpha=10, reg_lambda=10
3. Early stopping on validation set
4. Lower learning rate + more iterations (allows fine-tuning)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("LIGHTGBM WITH HEAVY REGULARIZATION")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/5] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

# Compress dtypes
def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

# Remove constant columns
const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

# Align columns
common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== NO ENGINEERED FEATURES ==============
print("\n[2/5] Using ORIGINAL FEATURES ONLY...")
X_train_feat = X_train.copy()
X_test_feat = X_test.copy()
print(f"✓ Features: {X_train_feat.shape[1]} (original)")

# ============== TRAIN LIGHTGBM ==============
print("\n[3/5] Training LightGBM with heavy regularization (3 seeds)...")

seeds = [42, 123, 456]
ensemble_preds = np.zeros(len(X_test_feat))
all_scores = []

# LightGBM parameters optimized for regularization
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 12,           # REDUCED (default=31, more like depth=5-6 in XGBoost)
    'learning_rate': 0.02,      # Lower learning rate (vs 0.05 in XGBoost)
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'max_depth': 5,             # Limit tree depth
    'reg_alpha': 10.0,          # Strong L1 regularization
    'reg_lambda': 10.0,         # Strong L2 regularization
    'min_data_in_leaf': 5,      # Minimum samples in leaf (prevent overfitting leaves)
    'verbose': -1,
    'num_threads': 4
}

for seed_idx, seed in enumerate(seeds, 1):
    print(f"\n  Model {seed_idx}/3 (seed={seed}):")
    
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    fold_indices = list(kf.split(X_train_feat))
    
    test_preds = np.zeros(len(X_test_feat))
    fold_scores = []
    
    for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
        X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        model = lgb.train(
            lgb_params,
            train_data,
            num_boost_round=1000,
            valid_sets=[val_data],
            callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=0)]
        )
        
        test_preds += model.predict(X_test_feat) / 5
        r2 = r2_score(y_val, model.predict(X_val))
        fold_scores.append(r2)
        
        del model, train_data, val_data, X_tr, X_val, y_tr, y_val
        gc.collect()
    
    avg_r2 = np.mean(fold_scores)
    all_scores.append(avg_r2)
    ensemble_preds += test_preds / 3
    print(f"    Avg R² (5-fold): {avg_r2:.6f}")
    
print(f"\n✓ Ensemble trained (3 models averaged)")
print(f"✓ Individual model scores: {[f'{s:.6f}' for s in all_scores]}")
print(f"✓ Ensemble Avg R²: {np.mean(all_scores):.6f}")

# ============== CREATE SUBMISSION ==============
print("\n[4/5] Creating submission...")
submission = pd.DataFrame({'ID': test_ids, 'TARGET': ensemble_preds})
print(f"✓ Submission shape: {submission.shape}")
print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")

# ============== SAVE SUBMISSION ==============
print("\n[5/5] Saving submission...")
submission.to_csv('submission_lightgbm_regularized.csv', index=False)
print(f"✓ Saved: submission_lightgbm_regularized.csv")

print("\n" + "="*70)
print("LIGHTGBM - READY FOR KAGGLE")
print("="*70)
print(f"✓ Strategy: LightGBM with heavy regularization")
print(f"✓ Advantages: Better for large datasets, leaf-wise growth, early stopping")
print(f"✓ Regularization: num_leaves=12, reg_alpha=10, reg_lambda=10")
print(f"✓ Features: Original 445 only")
print("="*70)